# Compare saved models in Jupyter

This notebook loads runs from `examples/v4/train.py` and computes a new gradient norm on one shared MNIST batch. Launch it inside the experiment's Git repository, with its `.flor` database and object store available. Install the example's PyTorch and torchvision dependencies in this kernel.

For your own experiment, replace `make_model` and the evaluation batch. The architecture must match each saved run. Checkpoint loading returns saved state and does not execute training code.

In [ ]:
from pathlib import Path

import pandas as pd
import torch
from torch import nn
from torchvision import datasets, transforms

import flordb as flor

runs = flor.dataframe()
if "hidden" not in runs.columns:
    raise RuntimeError("No v4 training runs found. Run examples/v4/train.py in this project first.")
runs = runs.loc[
    runs.source.eq("forward") & runs.filename.eq("train.py") & runs.hidden.notna()
].drop_duplicates("tstamp").copy()
if runs.empty:
    raise RuntimeError("No matching forward runs found.")
runs

Inspect checkpoint names before loading. New forward runs expose a `run` snapshot called `ckpt.pth`. Older runs may have only `iteration` snapshots: select their exact filenames explicitly. Missing files must be copied from the training machine; Git and `flor unpack` do not recover checkpoint bytes.

In [ ]:
flor.checkpoints(runs.iloc[0].tstamp)

The following model definition matches `examples/v4/train.py`, including its state dictionary keys. The recorded `hidden` argument selects the width for each run. If historical code changed more than the width, adapt this factory to construct those architectures too.

In [ ]:
class NeuralNet(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.fc1 = nn.Linear(784, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, 10)

    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))


def make_model(row):
    return NeuralNet(int(row.hidden))

Set `data_root` to the MNIST directory used by training. This cell uses already downloaded test data. Every model sees the same first 32 examples. `eval()` disables training behavior while still allowing gradients; gradients must be computed anew because they are absent from `state_dict()`.

In [ ]:
data_root = Path("../data")  # adjust to your existing MNIST directory
checkpoint_name = "ckpt.pth"

dataset = datasets.MNIST(
    root=str(data_root), train=False, download=False, transform=transforms.ToTensor()
)
loader = torch.utils.data.DataLoader(dataset, batch_size=32, shuffle=False)
inputs, targets = next(iter(loader))
inputs = inputs.reshape(-1, 784).cpu()
targets = targets.cpu()
loss_fn = nn.CrossEntropyLoss()

In [ ]:
measurements = []
for row in runs.itertuples(index=False):
    checkpoint = flor.load_checkpoint(row.tstamp, checkpoint_name)
    model = make_model(row).cpu()
    model.load_state_dict(checkpoint["model"])
    model.eval()
    model.zero_grad(set_to_none=True)
    loss = loss_fn(model(inputs), targets)
    loss.backward()
    grad_norm = sum(
        p.grad.detach().double().square().sum()
        for p in model.parameters() if p.grad is not None
    ).sqrt().item()
    measurements.append({
        "tstamp": row.tstamp,
        "comparison_loss": loss.item(),
        "grad_norm": grad_norm,
    })

comparison = runs.merge(pd.DataFrame(measurements), on="tstamp", validate="one_to_one")
comparison.sort_values("grad_norm")

The comparison remains in this notebook's dataframe. Loading checkpoints does not start another Flor run or write replay observations. The metric is a new measurement on the chosen evaluation batch, not a recovery of gradients from the original training step.

For a flat `torch.save(model.state_dict(), ...)` file, pass the loaded dictionary directly to `model.load_state_dict` instead of selecting `checkpoint["model"]`.